# LeetCode 3892 — Minimum Operations to Achieve At Least K Peaks bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-achieve-at-least-k-peaks/
- Độ khó: **Hard**
- Chủ đề: Mảng vòng, quy hoạch động "chọn k phần tử không kề nhau", tham lam có hối tiếc (regret greedy) với heap + danh sách liên kết, WQS binary search (Aliens trick)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, nums: list[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Chứng minh: chỉ cần **tăng các vị trí được chọn làm đỉnh**, và chi phí của mỗi đỉnh **độc lập** với nhau ⇒ bài toán trở thành **chọn `k` vị trí không kề nhau trên vòng tròn với tổng chi phí nhỏ nhất**.
2. Viết DP `O(n·k)` cho chuỗi rồi **cắt vòng** thành hai trường hợp.
3. Hiểu và cài đặt **tham lam có hối tiếc**: chọn phần tử rẻ nhất, thay nó bằng "phương án đổi ý" `c_L + c_R - c_i`, xóa hai láng giềng — `O(n log n)`.
4. Biết kỹ thuật **WQS binary search / Aliens trick** khi hàm chi phí theo `k` là **lồi**.
5. Kiểm thử bằng vét cạn thật sự trên **số lần tăng** (không dựa vào phép quy đổi).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng **vòng** `nums` độ dài `n`. Vị trí `i` là **đỉnh** nếu `nums[i]` **lớn hơn hẳn** hai láng giềng (trên vòng: láng giềng của `0` là `n-1` và `1`; của `n-1` là `n-2` và `0`). Mỗi thao tác: chọn một vị trí và **tăng 1**. Tìm số thao tác ít nhất để có **ít nhất `k` đỉnh**; nếu không thể, trả về `-1`.

**Ví dụ 1:** `nums = [2,1,2], k = 1` → `1` (tăng `nums[2]` lên 3).

**Ví dụ 2:** `nums = [4,5,3,6], k = 2` → `0` (đã có đỉnh tại 1 và 3).

**Ví dụ 3:** `nums = [3,7,3], k = 2` → `-1`.

**Ràng buộc:** `2 ≤ n ≤ 5000`, `-10^5 ≤ nums[i] ≤ 10^5`, `0 ≤ k ≤ n`.

In [1]:
import heapq, random, time
from itertools import product

examples = [([2, 1, 2], 1, 1), ([4, 5, 3, 6], 2, 0), ([3, 7, 3], 2, -1)]
for nums, k, exp in examples:
    print(nums, "k =", k, "-> kỳ vọng", exp)

[2, 1, 2] k = 1 -> kỳ vọng 1
[4, 5, 3, 6] k = 2 -> kỳ vọng 0
[3, 7, 3] k = 2 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Chỉ số vòng

Láng giềng trái của `i` là `nums[i - 1]` (Python tự hiểu `nums[-1]` là phần tử cuối), láng giềng phải là `nums[(i + 1) % n]`. Hai vị trí kề nhau trên vòng không thể **cùng** là đỉnh (mỗi cái phải lớn hơn cái kia) ⇒ tối đa `⌊n/2⌋` đỉnh.

In [2]:
def peaks(a):
    n = len(a)
    return [i for i in range(n) if a[i] > a[i - 1] and a[i] > a[(i + 1) % n]]

print(peaks([4, 5, 3, 6]), peaks([2, 1, 2]), peaks([1, 2]))

[1, 3] [] [1]


### 3.2. DP "chọn các phần tử không kề nhau" trên chuỗi

Kiểu *House Robber* (LC 198): `take[j]`, `skip[j]` = chi phí nhỏ nhất khi đã chọn `j` phần tử và phần tử cuối **được chọn / không được chọn**. Trên **vòng** (LC 213 *House Robber II*): tách thành hai chuỗi tùy theo phần tử đầu có được chọn hay không.

In [3]:
def rob_chain(vals):
    take, skip = float('-inf'), 0
    for v in vals:
        take, skip = skip + v, max(take, skip)
    return max(take, skip)

print(rob_chain([2, 7, 9, 3, 1]))       # 2 + 9 + 1 = 12

12


### 3.3. Heap + danh sách liên kết đôi (mảng `L`, `R`)

- `heapq` lấy phần tử nhỏ nhất `O(log n)`; phần tử "đã bị xóa" xử lý bằng **xóa lười** (đánh dấu `dead[i]`, bỏ qua khi pop).
- Danh sách liên kết đôi bằng hai mảng `L[i]`, `R[i]` cho phép xóa một nút `O(1)`: `R[L[x]] = R[x]`, `L[R[x]] = L[x]`.

In [4]:
n = 5
L = [(i - 1) % n for i in range(n)]; R = [(i + 1) % n for i in range(n)]
def remove(x):
    R[L[x]] = R[x]; L[R[x]] = L[x]
remove(2)
x, order = 0, []
for _ in range(4):
    order.append(x); x = R[x]
print("duyệt vòng sau khi xóa 2:", order)

duyệt vòng sau khi xóa 2: [0, 1, 3, 4]


### 3.4. Hàm lồi và "giá phạt" λ (ý tưởng WQS)

Nếu `f(k)` (chi phí tốt nhất khi chọn đúng `k`) là **lồi** (hiệu `f(k+1) - f(k)` không giảm), ta có thể bỏ ràng buộc "đúng `k`" bằng cách **trừ λ** cho mỗi phần tử được chọn và giải bài không ràng buộc; điều chỉnh λ bằng tìm nhị phân để số phần tử chọn được là `k`.

In [5]:
f = [0, 1, 3, 6, 10]                           # một dãy lồi
print("hiệu:", [f[i + 1] - f[i] for i in range(len(f) - 1)])
for lam in [0, 2, 3, 5]:
    best_k = min(range(len(f)), key=lambda k: (f[k] - lam * k, -k))
    print(f"λ={lam}: k tối ưu khi trừ λ mỗi phần tử = {best_k}")

hiệu: [1, 2, 3, 4]
λ=0: k tối ưu khi trừ λ mỗi phần tử = 0
λ=2: k tối ưu khi trừ λ mỗi phần tử = 2
λ=3: k tối ưu khi trừ λ mỗi phần tử = 3
λ=5: k tối ưu khi trừ λ mỗi phần tử = 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Chi phí làm `i` thành đỉnh** (nếu láng giềng giữ nguyên): `c_i = max(0, max(nums[i-1], nums[i+1]) + 1 - nums[i])`.

Quan sát:

1. **Không bao giờ nên tăng vị trí không phải đỉnh** — tăng nó chỉ làm láng giềng khó thành đỉnh hơn.
2. Các đỉnh **không kề nhau**, nên láng giềng của một đỉnh không phải đỉnh ⇒ không bị tăng ⇒ chi phí `c_i` **độc lập** giữa các đỉnh.
3. Vì `c_i ≥ 0`, chọn **đúng `k`** đỉnh là đủ. Nếu `k > ⌊n/2⌋` ⇒ `-1`.

Tính tay ví dụ 1, `[2,1,2]`: `c = [max(0, max(2,1)+1-2), max(0, 2+1-1), max(0, max(1,2)+1-2)] = [1, 2, 1]`. Chọn 1 vị trí rẻ nhất ⇒ `1` ✔.

Ví dụ 2, `[4,5,3,6]`: `c = [3, 0, 4, 0]`; hai vị trí không kề nhau rẻ nhất: `{1, 3}` ⇒ `0` ✔.

In [6]:
def costs(nums):
    n = len(nums)
    return [max(0, max(nums[i - 1], nums[(i + 1) % n]) + 1 - nums[i]) for i in range(n)]

for nums, k, exp in examples:
    print(nums, "-> chi phí từng vị trí:", costs(nums))

[2, 1, 2] -> chi phí từng vị trí: [1, 2, 1]
[4, 5, 3, 6] -> chi phí từng vị trí: [3, 0, 4, 0]
[3, 7, 3] -> chi phí từng vị trí: [5, 0, 5]


## 5. Cách 1 — Brute Force: vét cạn số lần tăng (không dùng quy đổi)

Để **kiểm chứng** quan sát ở mục 4, ta vét cạn thật sự: thử mọi vector số lần tăng `(t_0, …, t_{n-1})` với tổng `≤ B`, theo thứ tự tổng tăng dần; trả về tổng nhỏ nhất làm số đỉnh `≥ k`. Chỉ chạy được với `n` và giá trị rất nhỏ.

In [7]:
def brute_increments(nums, k, B=8):
    n = len(nums)
    if k > n // 2:
        return -1
    best = None
    for t in product(range(B + 1), repeat=n):
        s = sum(t)
        if best is not None and s >= best:
            continue
        a = [x + d for x, d in zip(nums, t)]
        if len(peaks(a)) >= k:
            best = s
    return best if best is not None else -1

for nums, k, exp in examples:
    print(brute_increments(nums, k, B=6), "kỳ vọng", exp)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


**Nút thắt:** số vector `(B+1)^n` bùng nổ. Hơn nữa chẳng biết chặn `B` bao nhiêu. Quy đổi ở mục 4 biến bài thành **chọn tập** — vẫn còn `C(n, k)` tập, cần DP hoặc tham lam.

In [8]:
# Vét cạn trên TẬP đỉnh (dùng quy đổi): duyệt mọi tập không kề nhau trên vòng
def brute_sets(nums, k):
    n = len(nums)
    if k == 0:
        return 0
    if k > n // 2:
        return -1
    c = costs(nums)
    best = None
    for mask in range(1 << n):
        if bin(mask).count('1') != k:
            continue
        if any(mask >> i & 1 and mask >> ((i + 1) % n) & 1 for i in range(n)):
            continue
        s = sum(c[i] for i in range(n) if mask >> i & 1)
        best = s if best is None else min(best, s)
    return best

for nums, k, exp in examples:
    print(brute_sets(nums, k), "kỳ vọng", exp)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


## 6. Ý tưởng thuật toán / bước đột phá

Sau quy đổi: **cho `c_0..c_{n-1} ≥ 0` trên vòng, chọn đúng `k` vị trí đôi một không kề nhau, tổng nhỏ nhất.**

**Hướng 1 — DP cắt vòng.** Hoặc vị trí 0 **không** được chọn ⇒ chuỗi `1..n-1` chọn `k`; hoặc vị trí 0 **được** chọn ⇒ không được chọn `1` và `n-1`, chuỗi `2..n-2` chọn `k-1`. Mỗi chuỗi giải bằng DP `O(n · k)`.

**Hướng 2 — Tham lam có hối tiếc (đột phá).** Lấy phần tử rẻ nhất `i` — nhưng có thể sau này ta muốn **đổi ý**: bỏ `i`, lấy cả hai láng giềng `L, R` thay thế. Việc đổi ý làm số phần tử tăng 1 với chi phí thêm `c_L + c_R - c_i`. Vậy:

> Sau khi chọn `i`: gán `c_i ← c_L + c_R - c_i` (nút "đổi ý"), **xóa** `L` và `R` khỏi vòng, nối `i` với láng giềng ngoài của chúng, đẩy lại vào heap.

Lặp `k` lần, cộng các giá trị lấy ra. Mỗi lần lấy ra một nút "đổi ý" chính là thực hiện việc thay `i` bằng `L, R` (các nút đổi ý có thể lồng nhau). Độ phức tạp `O(n log n)`.

**Hướng 3 — WQS.** `f(k)` lồi (chính tham lam ở trên cho thấy các chi phí biên lấy ra không giảm). Trừ `λ` cho mỗi phần tử chọn, giải DP `O(n)` không giới hạn số lượng, tìm nhị phân `λ`.

## 7. Các cách tiếp theo

- **Cách 2:** DP `O(n · k)` với cắt vòng.
- **Cách 3:** tham lam có hối tiếc, heap + danh sách liên kết — `O(n log n)` (lời giải nộp chính).
- **Cách 4:** WQS binary search + DP `O(n)` mỗi lần — `O(n log C)`.

### 7.1. Cách 2 — DP `O(n·k)` cắt vòng

**Giải thích:** `chain_best(vals, m)` trả về chi phí nhỏ nhất chọn **đúng** `m` phần tử không kề nhau trong chuỗi `vals` (hoặc `∞`). Mảng `skip[j]`, `take[j]` cập nhật theo từng phần tử; dùng list comprehension cho mỗi phần tử để giảm hằng số.

In [9]:
INF = float('inf')

def chain_best(vals, m):
    if m == 0:
        return 0
    if m < 0:
        return INF
    skip = [0] + [INF] * m          # skip[j]: đã chọn j, phần tử cuối KHÔNG chọn
    take = [INF] * (m + 1)          # take[j]: đã chọn j, phần tử cuối ĐƯỢC chọn
    for v in vals:
        new_take = [INF] + [skip[j - 1] + v for j in range(1, m + 1)]
        new_skip = [min(a, b) for a, b in zip(skip, take)]
        take, skip = new_take, new_skip
    return min(skip[m], take[m])

def dp_solution(nums, k):
    n = len(nums)
    if k == 0:
        return 0
    if k > n // 2:
        return -1
    c = costs(nums)
    case_skip0 = chain_best(c[1:], k)
    case_take0 = c[0] + chain_best(c[2:n - 1], k - 1)
    return min(case_skip0, case_take0)

for nums, k, exp in examples:
    print(dp_solution(nums, k), "kỳ vọng", exp)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


**Độ phức tạp Cách 2:** `O(n · k)` thời gian (≤ `5000 · 2500 · 2` ≈ `2,5·10^7` phép tính danh sách), `O(k)` bộ nhớ. Chạy được nhưng chậm trong Python ở ca xấu nhất.

In [10]:
class SolutionDP:
    def minOperations(self, nums: list[int], k: int) -> int:
        return dp_solution(nums, k)

print([SolutionDP().minOperations(a, k) for a, k, _ in examples])

[1, 0, -1]


### 7.2. Cách 3 — Tham lam có hối tiếc (lời giải chính)

**Giải thích từng đoạn:**

1. `val = costs(nums)`, `L`, `R` là danh sách liên kết vòng, `dead` đánh dấu nút đã xóa, heap chứa `(val[i], i)`.
2. Lặp `k` lần: pop đến khi gặp nút còn sống và giá trị khớp `val[i]` (bản ghi cũ bị bỏ qua). Cộng `val[i]`.
3. Nếu còn lượt sau: `l, r = L[i], R[i]`; `val[i] = val[l] + val[r] - val[i]`; xóa `l`, `r` (đánh dấu `dead`, nối `L[i] = L[l]`, `R[i] = R[r]` và cập nhật hai đầu); đẩy `(val[i], i)`.

Điều kiện `k ≤ ⌊n/2⌋` bảo đảm luôn còn đủ nút.

In [11]:
def regret_greedy(nums, k):
    n = len(nums)
    if k == 0:
        return 0
    if k > n // 2:
        return -1
    val = costs(nums)
    L = [(i - 1) % n for i in range(n)]
    R = [(i + 1) % n for i in range(n)]
    dead = [False] * n
    heap = [(v, i) for i, v in enumerate(val)]
    heapq.heapify(heap)
    total = 0
    for step in range(k):
        while True:
            v, i = heapq.heappop(heap)
            if not dead[i] and v == val[i]:
                break
        total += v
        if step == k - 1:
            break
        l, r = L[i], R[i]
        val[i] = val[l] + val[r] - val[i]
        dead[l] = dead[r] = True
        L[i] = L[l]; R[i] = R[r]
        R[L[i]] = i; L[R[i]] = i
        heapq.heappush(heap, (val[i], i))
    return total

for nums, k, exp in examples:
    print(regret_greedy(nums, k), "kỳ vọng", exp)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


**Độ phức tạp Cách 3:** mỗi bước một `heappop` (cộng các bản ghi cũ, tổng cộng `O(n + k)` bản ghi) và một `heappush` ⇒ `O((n + k) log n)`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — WQS binary search (Aliens trick)

**Giải thích:**

1. Với `λ` cho trước, giải bài **không giới hạn số lượng**: chọn tập không kề nhau tối thiểu hóa `Σ (c_i - λ)`; khi hòa thì ưu tiên **nhiều phần tử hơn**. DP vòng `O(n)` trả về `(giá_trị, số_lượng)`.
2. Số lượng tối ưu không giảm khi `λ` tăng. Tìm `λ` nhỏ nhất (nguyên) để số lượng `≥ k`.
3. Đáp án `f(k) = giá_trị(λ) + λ·k` (đúng nhờ tính lồi, kể cả khi có nhiều `k` cùng tối ưu tại `λ`).

Chi phí nguyên nên `λ` nguyên là đủ. **Cẩn thận chặn trên:** chi phí biên `f(k) - f(k-1)` có thể **lớn hơn `max(c)`** (khi phải "đổi ý" cả một khối), nên lấy `λ ∈ [0, Σc + 1]`.

In [12]:
def best_with_penalty(c, lam):
    """min Σ(c_i - lam) trên tập không kề nhau của VÒNG; hòa thì lấy nhiều phần tử. Trả (giá trị, số lượng)."""
    n = len(c)
    def chain(vals):
        # (giá trị, -số lượng) để min ưu tiên số lượng lớn
        skip, take = (0, 0), (INF, 0)
        for v in vals:
            nt = (skip[0] + v - lam, skip[1] - 1)
            ns = min(skip, take)
            take, skip = nt, ns
        return min(skip, take)
    a = chain(c[1:])                                   # 0 không chọn
    b = chain(c[2:n - 1]) if n >= 3 else (0, 0)        # 0 được chọn
    b = (b[0] + c[0] - lam, b[1] - 1)
    best = min(a, b)
    return best[0], -best[1]

def wqs_solution(nums, k):
    n = len(nums)
    if k == 0:
        return 0
    if k > n // 2:
        return -1
    c = costs(nums)
    lo, hi = 0, sum(c) + 1          # chi phí biên có thể vượt max(c) (đổi ý!), nên chặn trên bằng tổng
    while lo < hi:
        mid = (lo + hi) // 2
        if best_with_penalty(c, mid)[1] >= k:
            hi = mid
        else:
            lo = mid + 1
    value, _ = best_with_penalty(c, lo)
    return value + lo * k

for nums, k, exp in examples:
    print(wqs_solution(nums, k), "kỳ vọng", exp)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


In [13]:
class SolutionWQS:
    def minOperations(self, nums: list[int], k: int) -> int:
        return wqs_solution(nums, k)

print([SolutionWQS().minOperations(a, k) for a, k, _ in examples])

[1, 0, -1]


**Độ phức tạp Cách 4:** `O(n log(nC))` với `nC ≤ 5000 · 4·10^5` (khoảng 31 lần DP `O(n)`); bộ nhớ `O(1)` ngoài mảng chi phí.

## 8. TRACE từng bước

Trace Cách 3 trên vòng chi phí `c = [3, 1, 2, 5, 1, 4]` với `k = 3` (bỏ qua bước tính `c` từ `nums`).

| bước | lấy ra (i, val) | tổng | đổi ý: val[i] = val[L]+val[R]-val[i] | xóa | vòng còn lại |
|---|---|---|---|---|---|
| 1 | (1, 1) | 1 | 3 + 2 − 1 = 4 | 0, 2 | 1(4), 3(5), 4(1), 5(4) |
| 2 | (4, 1) | 2 | 5 + 4 − 1 = 8 | 3, 5 | 1(4), 4(8) |
| 3 | (1, 4) | 6 | – | – | – |

Bước 3 lấy nút "đổi ý" 1 (giá 4): nghĩa là bỏ vị trí 1, lấy 0 và 2 ⇒ tập `{0, 2, 4}` chi phí `3 + 2 + 1 = 6`. Vét cạn xác nhận.

In [14]:
def trace_regret(c, k):
    n = len(c); val = c[:]
    L = [(i - 1) % n for i in range(n)]; R = [(i + 1) % n for i in range(n)]
    dead = [False] * n
    heap = [(v, i) for i, v in enumerate(val)]; heapq.heapify(heap)
    total = 0
    for step in range(k):
        while True:
            v, i = heapq.heappop(heap)
            if not dead[i] and v == val[i]: break
        total += v
        msg = f"bước {step + 1}: lấy i={i} val={v} -> tổng={total}"
        if step < k - 1:
            l, r = L[i], R[i]
            val[i] = val[l] + val[r] - val[i]
            dead[l] = dead[r] = True
            L[i] = L[l]; R[i] = R[r]; R[L[i]] = i; L[R[i]] = i
            heapq.heappush(heap, (val[i], i))
            alive, x = [], i
            for _ in range(n):
                if (x, val[x]) in alive: break
                alive.append((x, val[x])); x = R[x]
            msg += f" | đổi ý val[{i}]={val[i]}, xóa {l},{r} | vòng: {alive}"
        print(msg)
    return total

c = [3, 1, 2, 5, 1, 4]
print("tham lam:", trace_regret(c, 3))
best = min(sum(c[i] for i in S) for S in __import__('itertools').combinations(range(6), 3)
           if all((j - i) % 6 not in (1, 5) for i in S for j in S if i != j))
print("vét cạn trên tập:", best)

bước 1: lấy i=1 val=1 -> tổng=1 | đổi ý val[1]=4, xóa 0,2 | vòng: [(1, 4), (3, 5), (4, 1), (5, 4)]
bước 2: lấy i=4 val=1 -> tổng=2 | đổi ý val[4]=8, xóa 3,5 | vòng: [(4, 8), (1, 4)]
bước 3: lấy i=1 val=4 -> tổng=6
tham lam: 6
vét cạn trên tập: 6


In [15]:
# Trace Cách 4: số phần tử chọn được theo λ (thấy tính đơn điệu)
for lam in range(0, 7):
    v, cnt = best_with_penalty(c, lam)
    print(f"λ={lam}: giá trị={v}, số phần tử={cnt}, f(cnt) ước lượng = {v + lam * cnt}")

λ=0: giá trị=0, số phần tử=0, f(cnt) ước lượng = 0
λ=1: giá trị=0, số phần tử=2, f(cnt) ước lượng = 2
λ=2: giá trị=-2, số phần tử=2, f(cnt) ước lượng = 2
λ=3: giá trị=-4, số phần tử=2, f(cnt) ước lượng = 2
λ=4: giá trị=-6, số phần tử=3, f(cnt) ước lượng = 6
λ=5: giá trị=-9, số phần tử=3, f(cnt) ước lượng = 6
λ=6: giá trị=-12, số phần tử=3, f(cnt) ước lượng = 6


## 9. Code nộp LeetCode

In [16]:
import heapq

class Solution:
    def minOperations(self, nums: list[int], k: int) -> int:
        n = len(nums)
        if k == 0:
            return 0
        if k > n // 2:
            return -1
        val = [max(0, max(nums[i - 1], nums[(i + 1) % n]) + 1 - nums[i]) for i in range(n)]
        L = [(i - 1) % n for i in range(n)]
        R = [(i + 1) % n for i in range(n)]
        dead = [False] * n
        heap = [(v, i) for i, v in enumerate(val)]
        heapq.heapify(heap)
        total = 0
        for step in range(k):
            while True:
                v, i = heapq.heappop(heap)
                if not dead[i] and v == val[i]:
                    break
            total += v
            if step == k - 1:
                break
            l, r = L[i], R[i]
            val[i] = val[l] + val[r] - val[i]
            dead[l] = dead[r] = True
            L[i] = L[l]
            R[i] = R[r]
            R[L[i]] = i
            L[R[i]] = i
            heapq.heappush(heap, (val[i], i))
        return total

sol = Solution()
for nums, k, exp in examples:
    got = sol.minOperations(nums, k)
    print(got, "✔" if got == exp else "✘")

1 ✔
0 ✔
-1 ✔


## 10. Vì sao thuật toán đúng

**Quy đổi.** Xét một lời giải tối ưu với tập đỉnh `P`, `|P| ≥ k`. Bỏ mọi lần tăng ở vị trí ngoài `P`: các đỉnh trong `P` vẫn là đỉnh (láng giềng chỉ giảm hoặc giữ). Mỗi đỉnh `i ∈ P` cần giá trị cuối `≥ max(nums[i-1], nums[i+1]) + 1` (láng giềng không đổi vì không thuộc `P` — hai đỉnh không kề nhau), tức cần ít nhất `c_i` lần tăng, và `c_i` lần là đủ. Vậy chi phí tối ưu = `min Σ_{i∈P} c_i` trên tập không kề nhau `|P| ≥ k`; do `c ≥ 0` nên lấy đúng `k`. Tập không kề nhau trên vòng có tối đa `⌊n/2⌋` phần tử ⇒ `k > ⌊n/2⌋` thì `-1`.

**Tham lam có hối tiếc.** Bất biến: sau `t` bước, tổng đã lấy bằng `f(t)` (tối ưu với `t` phần tử), và cấu trúc vòng hiện tại mô tả đúng "các thay đổi biên" có thể làm: lấy một nút gốc (thêm 1 phần tử) hoặc lấy một nút đổi ý (thay một khối `…` bằng khối xen kẽ ngược lại, cũng thêm đúng 1 phần tử). Đây là kết quả kinh điển (bài "trồng cây"/chọn k phần tử không kề nhau trên vòng); nó tương đương tìm đường tăng chi phí nhỏ nhất trong một luồng chi phí nhỏ nhất, nên các giá trị lấy ra **không giảm** và `f` lồi. Notebook kiểm chứng bằng vét cạn ở mục 12.

**WQS.** Vì `f` lồi, với mỗi `λ`, `min_k (f(k) - λk)` đạt tại các `k` mà `f(k) - f(k-1) ≤ λ ≤ f(k+1) - f(k)`. Chọn `λ` nhỏ nhất để số lượng tối ưu lớn nhất `≥ k` thì `k` nằm trong tập tối ưu, nên `f(k) = g(λ) + λk`.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `C` = chi phí lớn nhất (≤ `2·10^5 + 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn số lần tăng | `O((B+1)^n · n)` | `O(n)` |
| 1'. Vét cạn tập | `O(2^n · n)` | `O(n)` |
| 2. DP cắt vòng | `O(n · k)` | `O(k)` |
| 3. Tham lam có hối tiếc | `O(n log n)` | `O(n)` |
| 4. WQS | `O(n log(nC))` | `O(n)` |

## 12. Bộ test

In [17]:
def run_all(nums, k, with_brute=True):
    res = [dp_solution(nums, k), regret_greedy(nums, k), wqs_solution(nums, k), Solution().minOperations(nums, k)]
    if with_brute:
        res.append(brute_sets(nums, k))
    return res

for nums, k, exp in examples:
    assert all(v == exp for v in run_all(nums, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [18]:
# 12.1b Kiểm chứng PHÉP QUY ĐỔI bằng vét cạn số lần tăng (giá trị nhỏ)
rng = random.Random(1)
for _ in range(60):
    n = rng.randint(2, 4)
    nums = [rng.randint(0, 2) for _ in range(n)]
    k = rng.randint(0, n)
    assert brute_increments(nums, k, B=5) == Solution().minOperations(nums, k), (nums, k)
print("Quy đổi khớp vét cạn số lần tăng: OK")

Quy đổi khớp vét cạn số lần tăng: OK


In [19]:
# 12.2 ca biên (tự dựng)
cases = [
    ([5, 5], 1, 1),               # n = 2: tăng một phần tử lên 6
    ([5, 5], 2, -1),              # n = 2 không thể có 2 đỉnh
    ([1, 2, 3, 4], 0, 0),          # k = 0
    ([7, 7, 7, 7], 2, 2),          # vòng 4: chọn {0,2} hoặc {1,3}, mỗi cái tăng 1
    ([-10**5, 10**5, -10**5, 10**5], 2, 0),
    ([10**5, -10**5, 10**5, -10**5, 0], 2, 0),
]
for nums, k, exp in cases:
    r = run_all(nums, k)
    assert all(v == exp for v in r), (nums, k, r)
print("Ca biên: OK")

Ca biên: OK


In [20]:
# 12.3 phản ví dụ cho tham lam "lấy k chi phí nhỏ nhất không kề nhau theo thứ tự" (không có đổi ý)
def naive_greedy(nums, k):
    n = len(nums)
    if k > n // 2: return -1
    c = costs(nums); used = [False] * n; total = 0; cnt = 0
    for v, i in sorted((v, i) for i, v in enumerate(c)):
        if cnt == k: break
        if not used[i] and not used[i - 1] and not used[(i + 1) % n]:
            used[i] = True; total += v; cnt += 1
    return total if cnt == k else float('inf')

rng = random.Random(3)
for _ in range(5000):
    n = rng.randint(4, 8); nums = [rng.randint(0, 9) for _ in range(n)]; k = rng.randint(1, n // 2)
    if naive_greedy(nums, k) != Solution().minOperations(nums, k):
        print("phản ví dụ:", nums, "k =", k, "| chi phí:", costs(nums),
              "| tham lam thường:", naive_greedy(nums, k), "| đúng:", Solution().minOperations(nums, k))
        break

phản ví dụ: [9, 8, 2, 5, 9] k = 2 | chi phí: [1, 2, 7, 5, 1] | tham lam thường: 6 | đúng: 3


In [21]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3892)
for _ in range(500):
    n = rng.randint(2, 12)
    nums = [rng.randint(-20, 20) for _ in range(n)]
    k = rng.randint(0, n)
    r = run_all(nums, k)
    assert len(set(r)) == 1, (nums, k, r)
print("500 test nhỏ (có vét cạn tập): OK")
for _ in range(40):
    n = rng.randint(50, 300)
    nums = [rng.randint(-10**5, 10**5) for _ in range(n)]
    k = rng.randint(0, n // 2)
    r = run_all(nums, k, with_brute=False)
    assert len(set(r)) == 1
print("40 test vừa: OK")

500 test nhỏ (có vét cạn tập): OK


40 test vừa: OK


In [22]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(7)
n = 5000
nums = [rng.randint(-10**5, 10**5) for _ in range(n)]
for k in (1, 1000, 2500):
    t0 = time.time(); a = Solution().minOperations(nums, k); t1 = time.time()
    b = wqs_solution(nums, k)
    print(f"k={k}: đáp án {a}, khớp WQS: {a == b}, thời gian tham lam < 1s: {t1 - t0 < 1}")

k=1: đáp án 0, khớp WQS: True, thời gian tham lam < 1s: True


k=1000: đáp án 0, khớp WQS: True, thời gian tham lam < 1s: True
k=2500: đáp án 125554177, khớp WQS: True, thời gian tham lam < 1s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên tính vòng (coi như mảng thẳng)

In [23]:
def wrong_linear_costs(nums):
    n = len(nums)
    out = []
    for i in range(n):
        nb = [nums[j] for j in (i - 1, i + 1) if 0 <= j < n]      # SAI: đầu và cuối chỉ có 1 láng giềng
        out.append(max(0, max(nb) + 1 - nums[i]))
    return out

print("sai:", wrong_linear_costs([2, 1, 2]), "| đúng:", costs([2, 1, 2]))

sai: [0, 2, 0] | đúng: [1, 2, 1]


**Vì sao sai:** trên vòng, `nums[0]` kề `nums[-1]`. Ví dụ 1 sẽ ra `0` thay vì `1`.

### Lỗi 2 — Quên kiểm tra `k > ⌊n/2⌋`

Tham lam có hối tiếc chạy quá `⌊n/2⌋` bước sẽ lấy phải nút đã xóa / vòng suy biến, cho kết quả vô nghĩa thay vì `-1`.

In [24]:
def wrong_no_check(nums, k):
    n = len(nums); val = costs(nums)
    L = [(i - 1) % n for i in range(n)]; R = [(i + 1) % n for i in range(n)]
    dead = [False] * n; heap = [(v, i) for i, v in enumerate(val)]; heapq.heapify(heap); total = 0
    for step in range(k):
        while heap:
            v, i = heapq.heappop(heap)
            if not dead[i] and v == val[i]: break
        else:
            return "hết heap"
        total += v
        if step == k - 1: break
        l, r = L[i], R[i]; val[i] = val[l] + val[r] - val[i]
        dead[l] = dead[r] = True; L[i] = L[l]; R[i] = R[r]; R[L[i]] = i; L[R[i]] = i
        heapq.heappush(heap, (val[i], i))
    return total

print("sai:", wrong_no_check([3, 7, 3], 2), "| đúng:", Solution().minOperations([3, 7, 3], 2))

sai: 10 | đúng: -1


### Lỗi 3 — Không bỏ qua các nút đã bị xóa trong heap

Khi chọn `i`, hai láng giềng `l, r` bị xóa khỏi vòng nhưng bản ghi `(val[l], l)`, `(val[r], r)` **vẫn nằm trong heap**. Nếu không kiểm tra `dead`, ta có thể lấy lại một nút đã bị "nuốt" vào nút đổi ý — tức chọn hai vị trí kề nhau.

In [25]:
def wrong_stale(c, k):
    n = len(c); val = c[:]
    L = [(i - 1) % n for i in range(n)]; R = [(i + 1) % n for i in range(n)]
    dead = [False] * n; heap = [(v, i) for i, v in enumerate(val)]; heapq.heapify(heap); total = 0
    for step in range(k):
        while True:
            v, i = heapq.heappop(heap)
            if v == val[i]: break                    # SAI: thiếu kiểm tra dead[i]
        total += v
        if step == k - 1: break
        l, r = L[i], R[i]; val[i] = val[l] + val[r] - val[i]
        dead[l] = dead[r] = True; L[i] = L[l]; R[i] = R[r]; R[L[i]] = i; L[R[i]] = i
        heapq.heappush(heap, (val[i], i))
    return total

from itertools import combinations
rng = random.Random(33)
for _ in range(20000):
    n = rng.randint(4, 9); c = [rng.randint(0, 9) for _ in range(n)]; k = rng.randint(2, n // 2)
    correct = min(sum(c[i] for i in S) for S in combinations(range(n), k)
                  if all((j - i) % n not in (1, n - 1) for i in S for j in S if i != j))
    if wrong_stale(c, k) != correct:
        print("chi phí", c, "k =", k, "| sai:", wrong_stale(c, k), "| đúng:", correct)
        break

chi phí [2, 3, 4, 7, 8, 8, 2, 9] k = 4 | sai: 11 | đúng: 16


**Vì sao sai:** nút đã xóa có thể rẻ hơn nút đổi ý, nên bị lấy ra lần nữa — tổng nhỏ "ảo" vì tương ứng với một tập có hai vị trí kề nhau. **Sửa:** xóa lười — khi pop, bỏ qua nút có `dead[i]`.

*Ghi chú:* kiểm tra thêm `v == val[i]` là một thói quen an toàn của xóa lười; ở thuật toán này mỗi nút được cập nhật giá ngay sau khi chính bản ghi của nó vừa bị lấy ra, nên thực tế không có "bản ghi giá cũ" của nút còn sống — nhưng giữ kiểm tra này không tốn gì và chống lỗi khi sửa code.

### Lỗi 4 — Tăng cả láng giềng hoặc dùng chi phí "chênh lệch" có dấu

Viết `c_i = max(nums[i-1], nums[i+1]) + 1 - nums[i]` mà quên `max(0, …)` làm chi phí âm, tổng bị trừ đi một cách sai.

In [26]:
def wrong_negative_cost(nums, k):
    n = len(nums)
    c = [max(nums[i - 1], nums[(i + 1) % n]) + 1 - nums[i] for i in range(n)]   # SAI: thiếu max(0, ...)
    return min(sum(c[i] for i in S) for S in __import__('itertools').combinations(range(n), k)
               if all((j - i) % n not in (1, n - 1) for i in S for j in S if i != j))

print("sai:", wrong_negative_cost([4, 5, 3, 6], 2), "| đúng:", Solution().minOperations([4, 5, 3, 6], 2))

sai: -1 | đúng: 0


### Lỗi 5 — WQS với chặn trên `λ ≤ max(c)`

Trực giác "thêm một phần tử tốn tối đa `max(c)`" là **sai**: khi `k` gần `⌊n/2⌋`, muốn thêm một phần tử có thể phải dịch chuyển cả một khối (đổi ý), chi phí biên lớn hơn mọi `c_i`.

In [27]:
c_bad = [0, 22, 0, 18, 28, 0, 14, 15]
f = []
for kk in range(5):
    f.append(min(sum(c_bad[i] for i in S) for S in __import__('itertools').combinations(range(8), kk)
                 if all((j - i) % 8 not in (1, 7) for i in S for j in S if i != j)))
print("f(k) =", f, "| chi phí biên =", [f[i + 1] - f[i] for i in range(4)], "| max(c) =", max(c_bad))

f(k) = [0, 0, 0, 0, 42] | chi phí biên = [0, 0, 0, 42] | max(c) = 28


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Chỉ được tăng** + điều kiện "lớn hơn láng giềng" ⇒ chỉ tăng các vị trí được chọn; chi phí độc lập khi các vị trí không kề nhau.
- **Chọn `k` phần tử không kề nhau, tối ưu tổng** ⇒ DP `O(nk)`; khi `n, k` lớn ⇒ **tham lam có hối tiếc** (heap + danh sách liên kết, nút "đổi ý" `L + R - i`).
- **Mảng vòng** ⇒ cắt vòng thành hai trường hợp (phần tử đầu chọn / không chọn), hoặc dùng danh sách liên kết vòng.
- Hàm theo `k` **lồi** ⇒ cân nhắc WQS để bỏ chiều `k` khỏi DP.

## 15. Biến thể của bài

1. **Mảng thẳng** (không vòng): phần tử đầu/cuối chỉ có một láng giềng; tham lam có hối tiếc cần nút "lính canh" giá `∞` ở hai đầu.
2. **Đỉnh "không nhỏ hơn"** (`≥` thay vì `>`): chi phí `max(0, max(láng giềng) - nums[i])`.
3. **Được tăng hoặc giảm**: khi đó có thể giảm láng giềng, các chi phí không còn độc lập — bài khó hơn hẳn.
4. **Tối đa hóa tổng giá trị chọn `k` phần tử không kề nhau** (bài kinh điển "trồng cây"): cùng thuật toán với dấu ngược.

In [28]:
# Biến thể 4: tối đa hóa tổng k phần tử không kề nhau trên vòng = tham lam có hối tiếc với giá trị âm
def max_k_nonadjacent(a, k):
    n = len(a)
    if k > n // 2: return None
    val = [-x for x in a]
    L = [(i - 1) % n for i in range(n)]; R = [(i + 1) % n for i in range(n)]
    dead = [False] * n; heap = [(v, i) for i, v in enumerate(val)]; heapq.heapify(heap); total = 0
    for step in range(k):
        while True:
            v, i = heapq.heappop(heap)
            if not dead[i] and v == val[i]: break
        total += v
        if step == k - 1: break
        l, r = L[i], R[i]; val[i] = val[l] + val[r] - val[i]
        dead[l] = dead[r] = True; L[i] = L[l]; R[i] = R[r]; R[L[i]] = i; L[R[i]] = i
        heapq.heappush(heap, (val[i], i))
    return -total

from itertools import combinations
rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 10); a = [rng.randint(-10, 10) for _ in range(n)]; k = rng.randint(1, n // 2)
    best = max(sum(a[i] for i in S) for S in combinations(range(n), k)
               if all((j - i) % n not in (1, n - 1) for i in S for j in S if i != j))
    assert max_k_nonadjacent(a, k) == best
print("Biến thể 4 khớp vét cạn.")

Biến thể 4 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử số lần tăng / mọi tập | mũ | `O(n)` | Kiểm chứng quy đổi |
| 2. DP cắt vòng | House Robber có đếm số lượng | `O(nk)` | `O(k)` | Dễ viết, `n·k ≤ 10^7` |
| 3. Tham lam hối tiếc | Heap + liên kết, nút `L+R-i` | `O(n log n)` | `O(n)` | **Lời giải chuẩn** |
| 4. WQS | Phạt λ + DP `O(n)` | `O(n log(nC))` | `O(n)` | Khi biết hàm lồi, DP không giới hạn dễ viết |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [1, 1, 1, 1, 1, 1], k = 3` → ?

**Bài 2 (tính tay).** `nums = [5, 1, 4, 2, 3]`, `k = 2`: tính `c` và đáp án.

**Bài 3 (điền TODO).** Viết `costs_linear(nums)` cho biến thể **mảng thẳng**, trong đó vị trí đầu/cuối chỉ cần lớn hơn láng giềng duy nhất.

**Bài 4 (tự viết lại).** Viết lại Cách 2 nhưng chỉ dùng hai mảng 1 chiều và **không** tạo list mới mỗi phần tử (cập nhật `j` giảm dần).

**Bài 5 (mini challenge).** Biến thể mảng thẳng: tìm số thao tác ít nhất để có ≥ `k` đỉnh (đầu/cuối có thể là đỉnh nếu lớn hơn láng giềng duy nhất). Dùng DP và kiểm chứng vét cạn.

In [29]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def costs_linear(nums):
    # TODO: với i = 0 chỉ so với nums[1]; i = n-1 chỉ so với nums[n-2]
    pass

In [30]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_ops_linear(nums, k):
    # TODO: chain_best(costs_linear(nums), k), kiểm tra khả thi k <= (n+1)//2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi `c_i = 1`. (2) `k = 3 = ⌊6/2⌋`: chọn `{0,2,4}`. (3) Đáp án `3`.

**Bài 2.** (1) `c = [max(0, max(3,1)+1-5), max(0, 5+1-1), max(0, max(1,2)+1-4), max(0, 4+1-2), max(0, max(2,5)+1-3)] = [0, 5, 0, 3, 3]`. (2) Chọn `{0, 2}` không kề nhau. (3) Đáp án `0`.

**Bài 3.** (1) `n ≥ 2`. (2) Đầu: `max(0, nums[1] + 1 - nums[0])`. (3) Cuối tương tự.

**Bài 4.** (1) `take_new[j] = skip[j-1] + v`, `skip_new[j] = min(skip[j], take[j])`. (2) Cập nhật `j` giảm dần dùng biến tạm. (3) Đối chiếu `dp_solution`.

**Bài 5.** (1) Tối đa `⌈n/2⌉` đỉnh. (2) `chain_best` trên toàn bộ mảng. (3) Vét cạn `brute_increments` phiên bản mảng thẳng.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1, 2
assert Solution().minOperations([1, 1, 1, 1, 1, 1], 3) == 3
assert costs([5, 1, 4, 2, 3]) == [0, 5, 0, 3, 3]
assert Solution().minOperations([5, 1, 4, 2, 3], 2) == 0
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [32]:
# Bài 3
def costs_linear_sol(nums):
    n = len(nums); out = []
    for i in range(n):
        nb = [nums[j] for j in (i - 1, i + 1) if 0 <= j < n]
        out.append(max(0, max(nb) + 1 - nums[i]))
    return out

assert costs_linear_sol([2, 1, 2]) == [0, 2, 0]
print("Bài 3 OK")

Bài 3 OK


In [33]:
# Bài 4
def chain_best_inplace(vals, m):
    if m == 0: return 0
    skip = [0] + [INF] * m; take = [INF] * (m + 1)
    for v in vals:
        for j in range(m, 0, -1):
            t = skip[j - 1] + v
            skip[j] = min(skip[j], take[j])
            take[j] = t
        skip[0] = min(skip[0], take[0]); take[0] = INF
    return min(skip[m], take[m])

def dp_inplace(nums, k):
    n = len(nums)
    if k == 0: return 0
    if k > n // 2: return -1
    c = costs(nums)
    return min(chain_best_inplace(c[1:], k), c[0] + chain_best_inplace(c[2:n - 1], k - 1))

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 15); nums = [rng.randint(-9, 9) for _ in range(n)]; k = rng.randint(0, n)
    assert dp_inplace(nums, k) == Solution().minOperations(nums, k)
print("Bài 4 OK")

Bài 4 OK


In [34]:
# Bài 5
def min_ops_linear_sol(nums, k):
    n = len(nums)
    if k == 0: return 0
    if k > (n + 1) // 2: return -1
    return chain_best(costs_linear_sol(nums), k)

def brute_linear(nums, k, B=5):
    n = len(nums); best = None
    def lin_peaks(a):
        return sum(1 for i in range(n) if all(a[i] > a[j] for j in (i - 1, i + 1) if 0 <= j < n))
    for t in product(range(B + 1), repeat=n):
        s = sum(t)
        if best is not None and s >= best: continue
        if lin_peaks([x + d for x, d in zip(nums, t)]) >= k: best = s
    return -1 if best is None else best

rng = random.Random(5)
for _ in range(60):
    n = rng.randint(2, 4); nums = [rng.randint(0, 2) for _ in range(n)]; k = rng.randint(0, n)
    assert min_ops_linear_sol(nums, k) == brute_linear(nums, k), (nums, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
c = [max(0, max(a[i-1], a[(i+1)%n]) + 1 - a[i]) for i in range(n)]
if k == 0: return 0
if k > n // 2: return -1
# tham lam có hối tiếc trên vòng
L, R, dead, heap = ...
for step in range(k):
    pop tới khi (not dead[i] and v == val[i]); total += v
    if step < k-1:
        l, r = L[i], R[i]; val[i] = val[l] + val[r] - val[i]
        xóa l, r; nối i với L[l], R[r]; push (val[i], i)
```

**Checklist:**

- [ ] Láng giềng tính **theo vòng** (`nums[i-1]`, `nums[(i+1)%n]`)?
- [ ] Chi phí có `max(0, …)`?
- [ ] `k == 0` → `0`; `k > n//2` → `-1`?
- [ ] Heap bỏ qua bản ghi cũ (`v == val[i]`) và nút đã xóa?
- [ ] Sau khi xóa `l, r`, nối lại **hai chiều** (`R[L[i]] = i`, `L[R[i]] = i`)?